In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "dslim/bert-base-NER"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

print("NER model loaded successfully!")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER model loaded successfully!


In [9]:
def extract_entities(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0]

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities = []
    current_entity = None

    for token, prediction in zip(tokens, predictions):

        label = model.config.id2label[prediction.item()]

        # Ignore special tokens
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        # Join WordPiece subwords
        if token.startswith("##"):
            if current_entity:
                current_entity["word"] += token[2:]
            continue

        # New entity
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)

            current_entity = {
                "word": token,
                "label": label[2:]
            }

        # Continue existing entity
        elif label.startswith("I-") and current_entity:
            current_entity["word"] += " "

            current_entity["word"] += token

        # Outside entity
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    # Add final entity
    if current_entity:
        entities.append(current_entity)

    return entities

In [11]:
text = """
Srinivas is working on an AI project in Hyderabad.
He plans to visit Microsoft offices in Bengaluru next month.
"""

entities = extract_entities(text)

for entity in entities:
    print(f"{entity['word']:20} → {entity['label']}")

Srinivas             → PER
AI                   → MISC
Hyderabad            → LOC
Microsoft            → ORG
Bengaluru            → LOC


In [13]:
print("=" * 60)
print("🤖 AI NAMED ENTITY RECOGNIZER")
print("=" * 60)
print("Type 'exit' to stop.")

while True:
    text = input("\nEnter text: ")

    if text.strip().lower() == "exit":
        print("\nExiting NER.")
        break

    if not text.strip():
        print("Please enter some text.")
        continue

    entities = extract_entities(text)

    print("\n" + "-" * 60)
    print("DETECTED ENTITIES")
    print("-" * 60)

    if not entities:
        print("No entities detected.")
    else:
        for entity in entities:
            print(f"{entity['word']:20} → {entity['label']}")

🤖 AI NAMED ENTITY RECOGNIZER
Type 'exit' to stop.

Enter text: Elon Musk founded SpaceX and Tesla. He lives in the United States.

------------------------------------------------------------
DETECTED ENTITIES
------------------------------------------------------------
Elon Musk            → PER
SpaceX               → ORG
Tesla                → ORG
United States        → LOC

Enter text: exit

Exiting NER.
